In [1]:
import os
import random
import torch
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

from collections import defaultdict
import pandas as pd

###############################################################################
# Arguments
###############################################################################
args = {
    "models": [
        "nlpaueb/sec-bert-base",
        "bert-base-uncased"
    ],
    "datasets": [
        {
            "name": "yelp_review_full",
            "config": None,
            "split": "train",
            "text_column": "text"
        },
        {
            "name": "wikitext",
            "config": "wikitext-2-raw-v1",
            "split": "train",
            "text_column": "text"
        },
        {
            "name": "ag_news",
            "config": None,
            "split": "train",
            "text_column": "text"
        },
    ],
    "max_texts": 1000,
    "batch_size": 64,
    "drift_strengths": [0.0, 0.25, 0.5, 0.75, 1.0],
    "pca_components": 2,
    "output_dir": "results_multidataset",
}

os.makedirs(args["output_dir"], exist_ok=True)

###############################################################################
# Utility Functions
###############################################################################
def batch_generator(data, batch_size=32):
    for i in range(0, len(data), batch_size):
        yield data[i : i + batch_size]

def extract_cls_embeddings(model, tokenizer, texts, device):
    encodings = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )
    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
    return cls_embeddings.cpu().numpy()

def introduce_gradual_drift(text_list, fraction_shuffle=0.5):
    new_texts = []
    for txt in text_list:
        words = txt.split()
        if len(words) < 2:
            new_texts.append(txt)
            continue
        k = int(len(words) * fraction_shuffle)
        if k < 1:
            new_texts.append(txt)
            continue
        indices = list(range(len(words)))
        random.shuffle(indices)
        shuffle_indices = indices[:k]
        to_shuffle = [words[i] for i in shuffle_indices]
        random.shuffle(to_shuffle)
        for i, idx in enumerate(shuffle_indices):
            words[idx] = to_shuffle[i]
        new_texts.append(" ".join(words))
    return new_texts

###############################################################################
# DriftDetector Class
###############################################################################
class DriftDetector:
    def __init__(self, model, tokenizer, device, batch_generator, args, pca_transform=None):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.batch_generator = batch_generator
        self.args = args
        self.pca_transform = pca_transform
        self.prototype = None

        # We'll store the prototype after each batch
        self.prototypes = []
        # We'll store the *entire time series* of cosine similarities
        self.cosine_scores = []

        # For the final scatter plot, let's store all embeddings
        # (both baseline and drift) if we want to visualize them later:
        self.all_embeddings = []

    def initialize_baseline(self, texts):
        embeddings = []
        for batch in self.batch_generator(texts, self.args['batch_size']):
            cls_emb = extract_cls_embeddings(self.model, self.tokenizer, batch, self.device)
            embeddings.append(cls_emb)
        all_embeddings = np.concatenate(embeddings, axis=0)
        if self.pca_transform is not None:
            all_embeddings = self.pca_transform.transform(all_embeddings)
        self.prototype = np.mean(all_embeddings, axis=0)
        self.prototypes.append(self.prototype)
        # Also store baseline embeddings for scattering
        self.all_embeddings.extend(all_embeddings)

    def detect_drifts(self, texts):
        for batch in tqdm(self.batch_generator(texts, self.args['batch_size']), leave=False):
            batch_embeddings = extract_cls_embeddings(self.model, self.tokenizer, batch, self.device)
            if self.pca_transform is not None:
                batch_embeddings = self.pca_transform.transform(batch_embeddings)

            mean_emb = batch_embeddings.mean(axis=0, keepdims=True)
            sim = cosine_similarity(mean_emb, [self.prototype])[0][0]
            self.cosine_scores.append(sim)

            # For scatter plot, store these drift embeddings
            self.all_embeddings.extend(batch_embeddings)

            self._update_prototype(batch_embeddings)

    def _update_prototype(self, batch_embeddings):
        # This is a simple exponential weighting approach
        delta = batch_embeddings - self.prototype
        distances = np.linalg.norm(delta, axis=1)
        weights = np.exp(-distances / 2.0)
        weighted_sum = np.sum(weights[:, None] * delta, axis=0)
        self.prototype += weighted_sum / np.sum(weights)
        self.prototypes.append(self.prototype)


###############################################################################
# 1) Data Collection
#    For each dataset + model + drift_strength, we'll store:
#      - final cosine similarity (no PCA, with PCA)
#      - all intermediate cosines (time series)
#      - the final or entire set of drifted embeddings for plotting
###############################################################################
def collect_data():
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    print("Using device:", device)

    results = {}

    for dataset_info in args["datasets"]:
        dataset_name  = dataset_info["name"]
        dataset_config = dataset_info["config"]
        dataset_split = dataset_info["split"]
        text_col      = dataset_info["text_column"]

        print(f"\n=== Loading dataset: {dataset_name} ===")
        ds = load_dataset(dataset_name, dataset_config, split=dataset_split)
        texts = list(ds[text_col])
        random.shuffle(texts)
        # Limit the number of texts if desired
        if args["max_texts"] > 0 and len(texts) > args["max_texts"]:
            texts = texts[: args["max_texts"]]

        half_point = len(texts) // 2
        baseline_texts = texts[:half_point]
        drift_texts    = texts[half_point:]

        for model_name in args["models"]:
            print(f"\n--- Using Model: {model_name} ---")
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModel.from_pretrained(model_name)
            model.to(device)
            model.eval()

            # Fit PCA on baseline embeddings:
            baseline_embs = []
            for b in batch_generator(baseline_texts, args["batch_size"]):
                emb_b = extract_cls_embeddings(model, tokenizer, b, device)
                baseline_embs.append(emb_b)
            baseline_embs = np.concatenate(baseline_embs, axis=0)

            pca = PCA(n_components=args["pca_components"])
            pca.fit(baseline_embs)

            # We'll store full details so we can make the 6 subplots later:
            # results[(dataset_name, model_name)] -> list of dicts with:
            #   {
            #       "drift_strength": ...,
            #       "pca": True/False,
            #       "time_series": [all cosines],
            #       "final_similarity": float,
            #       "all_embeddings": <2D array of final embeddings for scatter>,
            #       "baseline_final": <the final similarity at drift_strength=0, to compute deltas>
            #   }
            # We’ll store in a nested dict for convenience:
            key = (dataset_name, model_name)
            if key not in results:
                results[key] = []

            # For each drift strength:
            for drift_strength in args["drift_strengths"]:
                print(f"Simulating drift_strength={drift_strength} ...")
                drifted_texts = introduce_gradual_drift(drift_texts, fraction_shuffle=drift_strength)
                test_texts    = baseline_texts + drifted_texts

                # 1) No PCA
                detector_no_pca = DriftDetector(
                    model=model,
                    tokenizer=tokenizer,
                    device=device,
                    batch_generator=batch_generator,
                    args=args,
                    pca_transform=None
                )
                detector_no_pca.initialize_baseline(baseline_texts)
                detector_no_pca.detect_drifts(test_texts)
                final_sim_no_pca = detector_no_pca.cosine_scores[-1]

                # 2) PCA
                detector_pca = DriftDetector(
                    model=model,
                    tokenizer=tokenizer,
                    device=device,
                    batch_generator=batch_generator,
                    args=args,
                    pca_transform=pca
                )
                detector_pca.initialize_baseline(baseline_texts)
                detector_pca.detect_drifts(test_texts)
                final_sim_pca = detector_pca.cosine_scores[-1]

                results[key].append({
                    "drift_strength": drift_strength,
                    "pca": False,
                    "time_series": detector_no_pca.cosine_scores[:],
                    "final_similarity": final_sim_no_pca,
                    "all_embeddings": np.array(detector_no_pca.all_embeddings)
                })
                results[key].append({
                    "drift_strength": drift_strength,
                    "pca": True,
                    "time_series": detector_pca.cosine_scores[:],
                    "final_similarity": final_sim_pca,
                    "all_embeddings": np.array(detector_pca.all_embeddings)
                })

    print("\nData collection done!")
    return results

# Collect the data
all_results = collect_data()

/Users/jasper.bruin/miniconda3/envs/driftwatch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps

=== Loading dataset: yelp_review_full ===

--- Using Model: nlpaueb/sec-bert-base ---
Simulating drift_strength=0.0 ...


Simulating drift_strength=0.25 ...


Simulating drift_strength=0.5 ...


Simulating drift_strength=0.75 ...


...

Simulating drift_strength=0.25 ...


Simulating drift_strength=0.5 ...


Simulating drift_strength=0.75 ...


Simulating drift_strength=1.0 ...



Data collection done!


In [4]:
###############################################################################
# 2) Plotting: Reproduce 6 subplots, but with drift_strength on x-axis
###############################################################################
def plot_six_subplots(all_results):
    """
    For each (dataset, model), we'll create a 2x3 grid of subplots:

      1) Cosine Similarity vs. Drift Strength
      2) Rolling Mean & Std Dev across drift strengths
      3) Histogram of final similarities
      4) Scatter Plot of Embeddings (PCA space) for the largest drift strength
      5) Delta from Baseline Similarity
      6) No PCA vs PCA Cosine Similarity (line plot again, or direct compare)

    Because we now have only 5 drift strengths, "rolling mean/std" is not the same
    as a time-series rolling window. We'll just illustrate how to compute a small
    'moving window' across these 5 points to mimic the original style.
    """

    # Group by (dataset, model)
    for (dataset_name, model_name), runs in all_results.items():
        # runs is a list of dicts for each drift_strength x pca in {True,False}
        # Let's reshape them into two groups: pca=False, pca=True
        no_pca = [r for r in runs if r["pca"] is False]
        pca_   = [r for r in runs if r["pca"] is True]

        # Sort each by drift_strength
        no_pca = sorted(no_pca, key=lambda x: x["drift_strength"])
        pca_   = sorted(pca_,   key=lambda x: x["drift_strength"])

        # 1) x-values: drift_strength
        x_no_pca = [r["drift_strength"] for r in no_pca]
        y_no_pca = [r["final_similarity"] for r in no_pca]
        x_pca    = [r["drift_strength"] for r in pca_]
        y_pca    = [r["final_similarity"] for r in pca_]

        # 2) For "rolling mean & std dev": we'll do a simple window across drift strengths
        def rolling_mean_std(values, window=2):
            """Compute rolling mean & std for a small window across the array."""
            means, stds = [], []
            for i in range(len(values)):
                wstart = max(0, i - window + 1)
                slice_ = values[wstart:i+1]
                means.append(np.mean(slice_))
                stds.append(np.std(slice_))
            return np.array(means), np.array(stds)

        # We'll do that for no_pca time_series final values:
        y_no_pca_rm, y_no_pca_std = rolling_mean_std(y_no_pca, window=2)
        y_pca_rm,    y_pca_std    = rolling_mean_std(y_pca,    window=2)

        # 3) For histogram, we'll just combine final similarities from no_pca and pca
        #    This is not a big distribution, but we'll plot them anyway
        all_final_sims_no_pca = y_no_pca
        all_final_sims_pca    = y_pca

        # 4) For scatter plot, let's pick the largest drift strength's embeddings
        #    i.e., drift_strength=1.0
        #    We'll show baseline (strength=0.0) in blue vs drift=1.0 in orange
        baseline_run_no_pca = [r for r in no_pca if r["drift_strength"] == 0.0][0]
        drift_run_no_pca    = [r for r in no_pca if r["drift_strength"] == 1.0][0]
        #   Because we used pca_transform=None in no_pca, these are raw embeddings
        #   For an actual "PCA space" scatter, pick the pca=True runs.
        #   Let's do pca=True so we actually see 2D embeddings:
        baseline_run_pca = [r for r in pca_ if r["drift_strength"] == 0.0][0]
        drift_run_pca    = [r for r in pca_ if r["drift_strength"] == 1.0][0]

        # We'll do a big figure
        fig, axs = plt.subplots(2, 3, figsize=(15,8))
        axs = axs.flatten()

        # Subplot 1: Cosine Similarity vs Drift Strength
        axs[0].plot(x_no_pca, y_no_pca, marker='o', label='No PCA')
        axs[0].plot(x_pca,    y_pca,    marker='s', label='PCA')
        axs[0].set_title("Cosine Similarity vs Drift Strength")
        axs[0].set_xlabel("Drift Strength")
        axs[0].set_ylabel("Cosine Similarity")
        axs[0].legend()
        axs[0].grid(True, linestyle='--', alpha=0.5)

        # Subplot 2: Rolling Mean & Std Dev
        # We'll plot the rolling means plus shading for std dev
        axs[1].plot(x_no_pca, y_no_pca_rm, color='blue', label='No PCA - Rolling Mean')
        axs[1].fill_between(x_no_pca,
                            y_no_pca_rm - y_no_pca_std,
                            y_no_pca_rm + y_no_pca_std,
                            color='blue', alpha=0.2,
                            label='No PCA - Rolling Std')

        axs[1].plot(x_pca, y_pca_rm, color='orange', label='PCA - Rolling Mean')
        axs[1].fill_between(x_pca,
                            y_pca_rm - y_pca_std,
                            y_pca_rm + y_pca_std,
                            color='orange', alpha=0.2,
                            label='PCA - Rolling Std')
        axs[1].set_title("Rolling Mean and Std Dev of Cosine Similarity")
        axs[1].set_xlabel("Drift Strength")
        axs[1].set_ylabel("Cosine Similarity")
        axs[1].legend()
        axs[1].grid(True, linestyle='--', alpha=0.5)

        # Subplot 3: Histogram of final similarities
        bins = np.linspace(min(0.9, min(all_final_sims_no_pca + all_final_sims_pca)),
                           max(1.0, max(all_final_sims_no_pca + all_final_sims_pca)),
                           10)
        axs[2].hist(all_final_sims_no_pca, bins=bins, alpha=0.7, label='No PCA')
        axs[2].hist(all_final_sims_pca,    bins=bins, alpha=0.7, label='PCA')
        axs[2].set_title("Histogram of Final Similarities")
        axs[2].set_xlabel("Cosine Similarity")
        axs[2].set_ylabel("Frequency")
        axs[2].legend()

        # Subplot 4: Scatter Plot of Embeddings (PCA Space)
        # We'll show baseline vs. drift in 2D from the pca=True runs
        baseline_embs_2d = baseline_run_pca["all_embeddings"]
        drift_embs_2d    = drift_run_pca["all_embeddings"]

        axs[3].scatter(baseline_embs_2d[:, 0], baseline_embs_2d[:, 1], alpha=0.6, label="Baseline")
        axs[3].scatter(drift_embs_2d[:, 0],    drift_embs_2d[:, 1],    alpha=0.6, label="Drifted")
        axs[3].set_title("Scatter Plot (PCA Space)")
        axs[3].set_xlabel("PC1")
        axs[3].set_ylabel("PC2")
        axs[3].legend()

        # Subplot 5: Delta from Baseline Similarity
        # Baseline similarity is the final_similarity at drift_strength=0.0
        baseline_sim_no_pca = [r for r in no_pca if r["drift_strength"] == 0.0][0]["final_similarity"]
        baseline_sim_pca    = [r for r in pca_   if r["drift_strength"] == 0.0][0]["final_similarity"]
        delta_no_pca = [sim - baseline_sim_no_pca for sim in y_no_pca]
        delta_pca    = [sim - baseline_sim_pca    for sim in y_pca]

        axs[4].plot(x_no_pca, delta_no_pca, marker='o', label='No PCA')
        axs[4].plot(x_pca,    delta_pca,    marker='s', label='PCA')
        axs[4].axhline(0.0, color='gray', linestyle='--', alpha=0.7)
        axs[4].set_title("Delta from Baseline Similarity")
        axs[4].set_xlabel("Drift Strength")
        axs[4].set_ylabel("Delta (Cosine Similarity)")
        axs[4].legend()
        axs[4].grid(True, linestyle='--', alpha=0.5)

        # Subplot 6: No PCA vs PCA Cosine Similarity (but instead compare the drifted vs baseline within PCA and non-PCA)
        # Subplot 6: Scatter Plot of Final Similarity vs Drift Strength with Color Coding for PCA vs No PCA
        axs[5].scatter(x_no_pca, y_no_pca, color='blue', label='No PCA', alpha=0.6)
        axs[5].scatter(x_pca, y_pca, color='orange', label='PCA', alpha=0.6)
        axs[5].set_title("Scatter Plot: Final Similarity vs Drift Strength (Color-coded PCA vs No PCA)")
        axs[5].set_xlabel("Drift Strength")
        axs[5].set_ylabel("Cosine Similarity")
        axs[5].legend()
        axs[5].grid(True, linestyle='--', alpha=0.5)



        fig.suptitle(f"{dataset_name} | {model_name}", fontsize=16)
        fig.tight_layout()

        # Save figure
        model_name_safe = model_name.replace("/", "_")
        fname = f"{dataset_name}_{model_name_safe}_6subplots.png"
        save_path = os.path.join(args["output_dir"], fname)
        plt.savefig(save_path)
        plt.close()
        print(f"Saved 6-subplot figure: {save_path}")


# Finally, call plot_six_subplots to generate your multi-panel figures
plot_six_subplots(all_results)
print("All done.")

Saved 6-subplot figure: results_multidataset/yelp_review_full_nlpaueb_sec-bert-base_6subplots.png
Saved 6-subplot figure: results_multidataset/yelp_review_full_bert-base-uncased_6subplots.png
Saved 6-subplot figure: results_multidataset/wikitext_nlpaueb_sec-bert-base_6subplots.png
Saved 6-subplot figure: results_multidataset/wikitext_bert-base-uncased_6subplots.png
Saved 6-subplot figure: results_multidataset/ag_news_nlpaueb_sec-bert-base_6subplots.png
Saved 6-subplot figure: results_multidataset/ag_news_bert-base-uncased_6subplots.png
All done.
